# Part I: The two-dimensional classical Ising model

**Goals**

- 1. Express the partition function as an (infinite) tensor network.
- 2. Approximate it by finite-size environment tensors computed through CTMRG.
- 3. Compute ensamble averages of magnetization and specific heat, including automatic differentiation.
<br>
<br>
<br>

Complete the **TODO** functions in order. Each unfinished exercise raises
`NotImplementedError`; run the check and scan cells after filling its dependencies.

## I.1 2D Stat-mech partition function as a tensor network

We consider a ferromagnetic Ising model on a square lattice with $J=k_B=1$:

$$
\mathcal H=-\sum_{\langle ij\rangle}\sigma_i\sigma_j-H\sum_i\sigma_i,
\qquad \sigma_i\in\{+1,-1\},\qquad \beta=1/T.
$$

where ${\langle ij\rangle}$ denotes nearest-neighbour sites and $H$ is an external magnetic field.
For $N$ sites, the partition function is

$$
Z_N(\beta,H)=\sum_{\{\sigma\}}
\prod_{\langle ij\rangle}e^{\beta\sigma_i\sigma_j}
\prod_i e^{\beta H\sigma_i}.
$$
<br>

We construct the partition function by assembling a tensor network with the same square lattice geometry: 
* Each edge carries the Boltzmann weights in a *bond matrix* $h_{ss'}=e^{\beta ss'}$.
* Ising spins live on vertices. Each vertex carries a "copy" tensor $x_{s_1s_2s_3s_4}=\delta_{s_1s_2s_3s_4}$
and a local field weight $w_s=e^{\beta Hs}$.
<br>
<br>

```text
    |     |     |           |    |    |
  --x--h--x--h--x--  =>   --A----A----A--
    |     |     |           |    |    |
    h     h     h           |    |    |
    |     |     |           |    |    |
  --x--h--x--h--x--       --A----A----A--
    |     |     |           |    |    |
```

For $\beta\geq0$, split each *bond matrix* using the symmetric **matrix square root**
$q=h^{1/2}$, with $q q=h$. Absorb four half-bonds into each vertex:

$$
A_{uldr}(\beta,H)=\sum_{s=\pm1}q_{us}q_{ls}q_{sd}q_{sr}\,w_s.
$$
where we order the legs as **up, left, down, right** (`uldr`). The bond dimension is $D=2$;

Contracting all copies of $A$ gives $Z_N$. 



### I.1.1 Construct the site tensors

Build $A(\beta,H)$ with PyTorch and use float64 throughout.
Keep tensor operations in PyTorch so that we can differentiate them later.

**Exercises 1a–1c:** implement $q=$`sqrt_h`, fill the weight and contraction gaps in on-site tensor `Ising_A`, and implement boundary tensors at boundaries of the system `C_0` and `T_0`.
<br>
<br>

The Ising spin degree of freedom `sigma` and copy tensor `x` are defined below.

In [ ]:
# Provided: imports.
import numpy as np
import torch
import matplotlib.pyplot as plt

# Set default floating-point precision to float64.
torch.set_default_dtype(torch.float64)

# Set default device to CPU.
# Switch to 'cuda' (Nvidia) or 'mps' (Apple Silicon) if you want to use accelerators)
torch.set_default_device("cpu")

In [ ]:
sigma = torch.tensor([1.0, -1.0])
x = torch.zeros((2, 2, 2, 2))
x[0, 0, 0, 0] = x[1, 1, 1, 1] = 1.0


def sqrt_h(beta: float)->torch.Tensor:
    """Return the matrix square root of h[s, s'] = exp(beta*s*s').
    """

    # TODO 1a: Construct h and return its symmetric matrix square root.
    raise NotImplementedError("Exercise 1a: implement sqrt_h.")


def Ising_A(beta: float, H:float = None)->torch.Tensor:
    """Build the partition-function tensor with legs ordered (up, left, down, right).

                         |u(0)
                         sh
                         |U(1)                          u(0)
                         |U(0)                          |
    l(0)--sh--L(1) L(1)--x--R(3) R(1)--sh--r(0) = l(1)--A--r(3) 
                         |D(2)                          |
                         |D(1)                          d(2)
                         sh
                         |d(0)
    
    The number in brackets (i) indicates the position of the index in the tensor.
    The number outside of brackets indicates the label assigned to index when specifying contraction diagram.

    H is the physical magnetic field: the local weight is w_s = exp(beta*H*sigma).
    None selects zero field.
    """

    sh = sqrt_h(beta)
    field = torch.as_tensor( beta * (0.0 if H is None else H), )

    # TODO 1b: Contract the half-bonds, copy tensor, and weights into A[up, left, down, right].
    #          Use torch.einsum (https://docs.pytorch.org/docs/2.14/generated/torch.einsum.html)
    raise NotImplementedError("Exercise 1b: define the Ising tensor.")

    return A

### I.1.2 Check lattice symmetries

Verify that $A$ is unchanged by $90^\circ$ rotations and reflections.
The four permutations below act on its `uldr` legs.


In [ ]:
# Provided checks for Exercises 1a–1b.
beta_check = 1.0

A = Ising_A(beta_check)
assert A.shape == (2, 2, 2, 2)
for permutation in [(1, 2, 3, 0), (3, 0, 1, 2), (2, 1, 0, 3), (0, 3, 2, 1)]:
    torch.testing.assert_close(A, A.permute(permutation))
print("Square-lattice symmetries verified.")

### I.1.3 Boundary for open boundary conditions

To fully specify the model, we have to specify its boundary conditions.
Here, we consider **open boundary** conditions. Therefore a spin on the corner or an edge of the square lattice 
contributes to parition function as:

```text
    x_2--qq--... => C_0--q--...  and   ...--qq--x_3--qq--... => ...--q--T_0--q--...
    |               |                           |                       |
    q               q                           q                       q
    q               ...                         q                       ...
    |                                           |
    ...                                         ...

    The order of indices of T_0 is chosen to be [left, right, down]
```

where the corner/edge vertices carry the adjusted "copy" tensors $(x_2)_{s_1s_2}=\delta_{s_1s_2}$ and $(x_3)_{s_1s_2s_3}=\delta_{s_1s_2s_3}$ respectively.
Here, we can choose to
* pin the spins on the boundary to either up (1) or down (-1).
* keep the spins free.

Construct $C_0$ and $T_0$ by summing over the shared spin index.
Include the local field weight once per site, for both free and pinned spins.

In [ ]:
def boundary_tensors(beta: float, H:float= None, pin_to:int=None)->tuple[torch.Tensor]:
    """Build the boundary tensors for the tensor network of the Ising model on a square lattice
    with open boundary conditions.

    pin_to specifies which spin, 1 or -1, to pin at the boundary.
    """
    assert pin_to in (None, 1, -1), "pin_to must be None, 1, or -1"

    x_2 = torch.zeros((2, 2))
    x_3 = torch.zeros((2, 2, 2))
    field = torch.as_tensor( beta * (0.0 if H is None else H), )

    if pin_to == 1:
        x_2[0, 0] = 1.0
        x_3[0, 0, 0] = 1.0
        weights = torch.exp(field * sigma) * (sigma == 1)
    if pin_to == -1:
        # TODO 1c: Impose down-spin boundary conditions
        pass
    if pin_to is None:
        x_2[0, 0] = x_2[1, 1] = 1.0
        x_3[0, 0, 0] = x_3[1, 1, 1] = 1.0
        weights = torch.exp(field * sigma)

    # Include weights once in the sum over the shared spin, even for pinned spins.
    # C_0= TODO 1c: Construct the corner tensor from x_2, sqrt_h, and weights
    # T_0= TODO 1c: Construct the edge tensor from x_3, sqrt_h, and weights, with order of legs [left, right, down]
    raise NotImplementedError("Exercise 1c: define boundary tensors.")

    return C_0, T_0

## I.2 Approximate contraction of infinite TN: CTMRG

Evaluating partition function $Z_N$, represented as tensor network, by *exact* contraction is exponentially hard for large $N$.
The goal here is to employ real-space renormalization group approach to instead build a finite and controlled approximation of $Z_N$.

```text

Z_N = C_0--T_0--...--T_0--C_0  \approx  C*--T*--C*  
      |    |         |    |             |   |   |
      T_0--A--  ...  A----T_0           T*--A---T*
      |    |         |    |             |   |   |
      ...            ...                C*--T*--C*
      |    |         |    |
      T_0--A--  ...  A----T_0
      |    |         |    |
      C_0--T_0--...--T_0--C_0
```

where $C^*$ and $T^*$ are finite tensors, with sizes $\chi\times\chi$ and $\chi\times\chi\times D$ respectively. The $\chi$ is dubbed 
*environment* dimension and it will control the precision of approximation.

The renormalization process seeks a fixed point tensor $C^*$, $T^*$ of the map

```text
                       C---T---T---T---C
                       |   |   |   |   |
C--T--C => enlarge =>  T---A---A---A---T => truncate & absorb => C'--T'--C'
|  |  |                |   |   |   |   |                         |   |   |
T--A--T                T---A---A---A---T                         T'--A---T'
|  |  |                |   |   |   |   |                         |   |   |
C--T--C                T---A---A---A---T                         C'--T'--C'
                       |   |   |   |   |
                       C---T---T---T---C
```

The key step worked-out below is the truncation and absorption.


### I.2.1 Initialize the CTM boundary

Represent the initial environment by a corner matrix $C$ and an edge tensor $T$.
Square-lattice symmetry of the network and point-group symmetries of the on-site tensor `Ising_A` lets us use the same tensors in all four directions.

| Tensor | Shape | Meaning |
|---|---|---|
| $C$ | $(b,b)$ | Corner |
| $T$ | $(b,b,D)$ | Edge |

Here $b\leq\chi$ is the current boundary dimension; $\chi$ is the truncation cap.

```text
    C--1           0---T---1
    |                  |
    0                  2
```

Two starting environments:

- **Open boundary:** These are `C_0` and `T_0` from section I.1.3
- **Random boundary:** draw random entries and symmetrize $C$ and the first
  two legs of $T$. This gives reflection-symmetric starting tensors.

Random boundaries can select either magnetization branch below $T_c$.
We use them in the observable scans.

**Exercise 2a:** symmetrize the random boundaries; allocation and shapes are provided.

In [ ]:
def init_obc_CT(beta: float, H: float = None, pin_to: int = None)->tuple[torch.Tensor]:
    """Build open boundaries.

    Return C.shape == (2, 2) and T.shape == (2, 2, 2). Subsequent CTMRG
    updates grow the active dimension up to chi without padding zero states.
    """
    return boundary_tensors(beta, H, pin_to)


def init_rand_CT(A: torch.Tensor, chi: int)->tuple[torch.Tensor]:
    """Build random boundaries with C and each T[:, :, s] real and symmetric."""
    assert chi>0, "boundary tensors need dimension chi > 0"

    C = torch.rand((chi, chi))
    T = torch.rand((chi, chi, A.shape[0]))

    # TODO 2a: Symmetrize C and the first two legs of T.
    raise NotImplementedError("Exercise 2a: symmetric random boundaries.")

    return C, T


### I.2.2 Enlarge, truncate, and absorb

A **corner transfer matrix renormalization group (CTMRG)** step absorbs one
layer, then compresses the boundary back to at most $\chi$ states.

```text
    C---T---T---T---C
    |   |   |   |   |
    T---A---A---A---T
    |   |   |   |   |
    T---A---A---A---T
    |   |   |   |   |
    T---A---A---A---T
    |   |   |   |   |
    C---T---T---T---C
```

#### 1. Enlarge the corner

Contract one corner, two edges, and on-site tensor.
Pair the open legs to form a $(bD)\times(bD)$ matrix $C'$:

```text
    C---T--        C'==
    |   |     =    ||
    T---A--
    |   |
```

**Exercise 2b:** implement `enlarged_C` using the indexed diagram below.

In [ ]:
def enlarged_C(A, C, T)->torch.Tensor:
    r"""
    Build enlarged corner tensor

        C----b(1) b(0)---T--c(1) = C'==c(2),h(3)
        a(0)             d(2)      ||
        a(1)             d(0)      e(0),g(1)
        T--f(2) f(1)-----A--h(3)
        e(0)             g(2)

    """

    # TODO: Contract the enlarged corner using the indexed diagram.
    raise NotImplementedError("Exercise 2a: implement enlarged_C.")


#### 2. Truncate corner and update the edge

The best approximation of a matrix $M$ by a matrix $M'$ with a smaller rank at most $\chi$ in Frobenius norm (usual 2-norm) is solved by SVD. 
For real symmetric matrix, we can use diagonalization:

$$
C'=U'\operatorname{diag}(S')U'^T.
$$

Keep $r=\min(\chi,bD)$ eigenvectors with the largest eigenvalue magnitudes.
They form an isometry $U\in\mathbb R^{bD\times r}$, with $U^TU=I_r$.

The approximation $U\operatorname{diag}(S)U^T$ minimizes the Frobenius error
among rank-$r$ matrices. In the retained basis, the new corner is diagonal:

```text

    C--T--
    |  |  |U--  =  C_new--
    T--A--         |
    |__|
     U
     |
```

$$
C_{\mathrm{new}}=U^TC'U=\operatorname{diag}(S).
$$

For the edge, absorb one $A$ and compress both enlarged boundary legs using $U$.
The bulk leg remains open:

```text
              |
          ----U----            
          |       |             |
          T-------A--    =    T_new--
          |       |             |
          ----U----            
              |
```

The indexed diagram in `T_absorb` specifies the leg ordering.

**Exercise 2b:** fill the eigendecomposition and state-selection gaps.

**Exercise 2c:** implement the edge contraction in `T_absorb`.


In [ ]:
def compress(chi, enlarged_C)->tuple[torch.Tensor]:
    r"""
    Return the retained eigenvalues S and the truncation isometry U.

    The input has shape (b, D, b, D), where b is the current boundary
    dimension. Retain at most r = min(chi, b*D) eigenvectors,
    ordered by decreasing eigenvalue magnitude. S is a vector of eigenvalues, not a matrix.

    We reshape isometry U holding the leading r eigenvectors of C' into a rank-3 tensor with index order:

    (0) size b --
                 |U-- (2) size r
    (1) size D --
 
    """

    down_chi, down_D, right_chi, right_D = enlarged_C.shape
    matrix = enlarged_C.reshape(down_chi * down_D, right_chi * right_D)

    # TODO 2b: Diagonalize matrix and choose indices for the largest \chi (in magnitude) eigenvalues.
    raise NotImplementedError("Exercise 2b: corner eigenspace.")

    S = S[indices]
    U = U[:, indices].reshape(down_chi, down_D, len(indices))
    return S, U


def T_absorb(A, T, U)->torch.Tensor:
    r"""
    Absorb on-site tensor into the edge tensor and compress with the corner isometry.

        g(2)
        U
        |------------\
        b(0)          f(1)
        b(0)          f(0)      g(1)
        T--c(2) c(1)--A--h(3) = T--h(2)
        a(0)          e(2)      d(0)
        a(0)          e(1)
        |------------/
        U
        d(2)
    """

    # TODO 2c: Absorb A into T and project both enlarged boundary legs with U.
    raise NotImplementedError("Exercise 2c: implement T_absorb.")


### I.2.3 Compose one CTMRG step

Combine corner enlargement, truncation, and edge absorption.
Store $C_{\mathrm{new}}$ in its diagonal basis.

**Exercise 2d:** combine the three building blocks in `ctm_iter`.


In [ ]:
def ctm_iter(A, C, T, chi)->tuple[torch.Tensor]:
    r"""
    Perform one CTMRG step, retaining at most chi boundary states.
    """

    # TODO 2d: Combine enlargement, truncation, and edge absorption into one step.
    #          Given current C,T i)   compute enlarged corner
    #                            ii)  compute the isometry and truncated enlarged corner, embed truncated spectrum of C' into a matrix
    #                            iii) compute renormalized edge tensor T
    raise NotImplementedError("Exercise 2d: implement ctm_iter.")

    assert len(C_new.shape)==2 and len(T_new.shape)==3, "Unexpected shapes of C_new, T_new"
    return C_new, T_new

### I.2.4 Iterate steps forming a fixed-point algorithm

After each update:

1. Normalize $C$ and $T$ by their largest absolute entries.
2. Symmetrize the first two legs/indices of $T$ to suppress roundoff errors.
3. Compare consecutive normalized corner spectra, sorted by eigenvalue magnitude.

Stop when the summed absolute spectrum difference is below the tolerance.
If the iteration cap is reached, the environment may still be unconverged.

Scalar normalizations cancel in the observable ratios below.
Finite-$\chi$ errors remain: check larger $\chi$ near the transition.

**Exercise 4:** complete the three TODO blocks in the loop:

1. Advance one CTM step.
2. Normalize the corner.
3. Symmetrize and normalize the edge.

The convergence test, callback, and failure reporting are provided.
Keep the tensor updates outside `torch.no_grad()`.


In [ ]:
def ctmrg(
    A,
    C,
    T,
    chi,
    ctm_max_iter=100,
    ctm_conv_tol=1.0e-8,
    callback=None,
    raise_on_nonconvergence=False,
)->tuple[torch.Tensor]:
    """Iterate CTMRG, retaining at most chi states.

    Complete the three TODO blocks inside the loop.
    Convergence checks, callback handling, and failure reporting are provided.
    Keep the tensor updates outside no_grad so they remain differentiable.
    """

    # intial setup
    with torch.no_grad():
        spectrum = torch.linalg.eigvalsh(C)
        indices = torch.argsort(spectrum.abs(), descending=True)
        previous_spectrum = spectrum[indices] / spectrum.abs().max()

    for i in range(ctm_max_iter):
        # TODO 1: Perform one update using ctm_iter.
        raise NotImplementedError("Exercise 4: update C and T.")

        # TODO 2: Normalize C by its largest absolute entry.
        raise NotImplementedError("Exercise 4: normalize C.")

        # TODO 3: Symmetrize the first two legs of T, then normalize T.
        raise NotImplementedError("Exercise 4: symmetrize and normalize T.")

        
        # Tracking convergence: Compute the corner spectrum difference between iterations
        with torch.no_grad():
            spectrum = C.diagonal()
            indices = torch.argsort(spectrum.abs(), descending=True)
            spectrum = spectrum[indices]
            # Boundary dimensions may grow during the first few updates.
            size = max(spectrum.numel(), previous_spectrum.numel())
            current_padded = torch.nn.functional.pad(
                spectrum, (0, size - spectrum.numel())
            )
            previous_padded = torch.nn.functional.pad(
                previous_spectrum, (0, size - previous_spectrum.numel())
            )
            residual = (current_padded - previous_padded).abs().sum()
            converged = bool(residual < ctm_conv_tol)
            previous_spectrum = spectrum

        if callback is not None:
            finished = converged or i == ctm_max_iter - 1
            callback(i, residual, finished, A, C, T)
        if converged:
            break

    if raise_on_nonconvergence and not converged:
        raise RuntimeError(
            f"CTMRG did not converge in {ctm_max_iter} iterations: "
            f"residual={residual.item():.3e}, tolerance={ctm_conv_tol:.3e}."
        )
    return C, T


In [ ]:
# Sanity check away from criticality, with reproducible random boundaries.
torch.manual_seed(0)
beta_check = 0.2
chi_check = 8
A_check = Ising_A(beta_check)
C_check, T_check = init_rand_CT(A_check, chi_check)
C_check, T_check = ctmrg(
    A_check, C_check, T_check, chi_check,
    ctm_conv_tol=1e-10, raise_on_nonconvergence=True,
)

assert C_check.shape == (chi_check, chi_check)
assert T_check.shape == (chi_check, chi_check, A_check.shape[0])
assert torch.isfinite(C_check).all() and torch.isfinite(T_check).all()
print("CTMRG converged and returned finite tensors.")
print("C shape:", tuple(C_check.shape), "; T shape:", tuple(T_check.shape))
print("Normalized corner spectrum:", C_check.diagonal())

## I.3 Measuring ensamble averages

In general, measuring averages amounts to computing 

$$
\langle O(\sigma_a,...,\sigma_b) \rangle(\beta,H) = \frac{1}{Z(\beta,H)} \sum_{\{\sigma\}} O(\sigma_a,...,\sigma_b) e^{-\beta\mathcal{H}(\{\sigma\};H)}
$$

Below, we consider on-site magnetization in the center of the system, far away from all boundaries. The general expression reduces to

$$
m= \langle \sigma_0 \rangle(\beta,H) = \frac{1}{Z(\beta,H)} \sum_{\{\sigma\}} \sigma_0 e^{-\beta\mathcal{H}(\{\sigma\};H)}
$$

which corresponds to a slightly modified network

```text

    |     |            |           |    |    |
  --x--h--x--h---------x--  =>   --A----A----A--
    |     |            |           |    |    |
    h     h            h           |    |    |
    |     |            |           |    |    |
  --x--h--\sigma_0--h--x--       --A----B----A--
    |     |            |           |    |    |
    h     h            h           |    |    |
    |     |            |           |    |    |
  --x--h--x--h---------x--       --A----A----A-- 
    |     |            |           |    |    |
```

where $B$ is a new *impurity* tensor, carrying effect of spin. Our strategy will be to treat infinite parts of the network
by CTMRG, while accounting for impurity in the middle of the system.

### I.3.1 Measure the magnetization

Insert a factor of $s$ into the site tensor to obtain the impurity tensor $B$:

$$
B_{uldr}=\sum_{s=\pm1}q_{us}q_{ls}q_{sd}q_{sr}\,w_s\,s.
$$

Contract four corners and four edges into the one-site environment
$\rho_{uldr}$. The magnetization is the normalized contraction

$$
m=\langle\sigma_i\rangle
=\frac{\sum_{uldr}\rho_{uldr}B_{uldr}}
       {\sum_{uldr}\rho_{uldr}A_{uldr}}.
$$

```text
       C---T---C         C---T---C
       |   |   |         |   |   |
  m =  T---B---T    /    T---A---T
       |   |   |         |   |   |
       C---T---C         C---T---C
```

Use the same $\beta$ and $H$ for $A$ and $B$.

**Exercises 5a–5c:** fill the impurity weights and contractions, follow the three
environment-contraction stages, and implement the observable ratio.


In [ ]:
def f_B(beta, H=None):
    r"""
    Build the magnetization impurity tensor B at the same beta and H as A.
    """

    h = sqrt_h(beta)
    field = torch.as_tensor(beta * (0.0 if H is None else H))

    # TODO 5a: Include sigma in the local weights and contract the impurity tensor B.
    raise NotImplementedError("Exercise 5a: impurity tensor.")

    return B


def aux_rho_1x1(C, T):
    r"""
    Contract four corners and four edges into rho[up, left, down, right].

        i) C--b(1) b(0)--T--c(1) = C-----T--c(1)
           |             |         |     |
           a(0)          d(2)      a(0)  d(2)

        ii) C-----T--b(1) b(0)--C  =  C----T--------C
            |     |             |     |    |        |
            a(0)  c(2)          |     a(0) c(2)     |
                          e(2)--T             e(3)--T
                                |                   |
                              d(1)                  d(1) 

        iii) C------T--------C                C------T--------C
             a(0)   |        |                |      |        |
             a(1)   u(2)     |                |      u(0)     |
             T---l(3)  r(3)--T        =       T---l(1)  r(3)--T
             |      d(2)     b(1)             |      d(2)     |
             |      |        b(0)             |      |        |
             C------T--------C                C------T--------C
    """

    # TODO: Form rho using diagram (i).
    raise NotImplementedError("Exercise 5b: diagram i.")

    # TODO: Join the corner-edge blocks as in diagram (ii).
    raise NotImplementedError("Exercise 5b: diagram ii.")

    # TODO: Close the boundary as in diagram (iii), with the site legs ordered as uldr.
    raise NotImplementedError("Exercise 5b: diagram iii.")

    return rho


def eval_1x1(B, A, rho):
    r"""
    Return the normalized expectation value of B in the environment rho.
    """

    # TODO: Return the normalized impurity contraction.
    raise NotImplementedError("Exercise 5c: implement eval_1x1.")


### I.3.2 Compare with the exact magnetization

At zero field, the critical inverse temperature is

$$
\beta_c=\frac12\log(1+\sqrt2).
$$

The positive spontaneous-magnetization branch is

$$
m_{\mathrm{exact}}(\beta)=
\begin{cases}
0,&\beta\leq\beta_c,\\
\left[1-\sinh^{-4}(2\beta)\right]^{1/8},&\beta>\beta_c.
\end{cases}
$$

- A finite spin-symmetric system has $m=0$ at $H=0$.
- Spontaneous magnetization takes the thermodynamic limit before $H\to0^+$.
- Random boundaries select either sign, so compare $|m|$ with the positive branch.

Use a fixed seed for reproducibility. Check the final convergence residual at
each temperature.

**Provided scan:** run the next two cells after completing Exercises 1–5.


In [ ]:
# Provided: convergence monitor and magnetization scan.
def report_ctm(i, residual, finished, A, C, T):
    """Print the final iteration count and corner-spectrum residual."""
    if finished:
        print(f"iterations={i + 1}, residual={residual.item():.2e}")


torch.manual_seed(7)
chi = 12
betas = np.linspace(0.1, 0.6, 20)
magnetizations = []

with torch.no_grad():
    for beta in betas:
        print(f"beta={beta:.4f}", end=", ")
        A = Ising_A(beta)
        C, T = ctmrg(
            A,
            *init_rand_CT(A, chi),
            chi,
            ctm_max_iter=3000,
            ctm_conv_tol=1.0e-8,
            callback=report_ctm,
        )
        m = eval_1x1(f_B(beta), A, aux_rho_1x1(C, T))
        magnetizations.append(m.item())

In [ ]:
# Provided: exact magnetization and plotting helper.
def exact_magnetization(beta):
    """Return the positive spontaneous magnetization, including the disordered phase."""
    beta = np.asarray(beta, dtype=float)
    beta_c = np.log1p(np.sqrt(2)) / 2
    ordered = beta > beta_c
    result = np.zeros_like(beta)
    result[ordered] = (1 - np.sinh(2 * beta[ordered]) ** (-4)) ** (1 / 8)
    return result


def plot_magnetization(betas, values, title):
    """Compare |m| with the exact zero-field magnetization."""
    values = np.abs(np.asarray(values, dtype=float))
    beta_grid = np.linspace(betas.min(), betas.max(), 200)
    fig, ax = plt.subplots(1, 1, figsize=(5, 4), constrained_layout=True)
    ax.scatter(betas, values, label="CTMRG")
    ax.plot(beta_grid, exact_magnetization(beta_grid), color="black", label="Exact")
    ax.set(xlabel=r"$\beta$", ylabel=r"$|m|$")
    ax.legend()
    fig.suptitle(title)
    return fig, ax


fig_impurity, ax_impurity = plot_magnetization(
    betas, magnetizations, "Magnetization from an impurity tensor"
)
plt.show()


## I.4 [Extra] Free energy - partition function per site

The partition function per site $z$ and the associated free energy per site $f$ are

$$
z=\lim_{N\to\infty}Z_N^{1/N},\qquad
\log z=\lim_{N\to\infty}\frac{\log Z_N}{N},\qquad
f=-\frac{\log z}{\beta},
$$

where $N$ is the total number of sites.

### I.4.1. Free-energy per site from scale-invariant ratio

A single patch contraction depends on the normalization of $C$ and $T$,
which are rescaled at every CTMRG step.
To cancel these scale factors, form four scalars from the **same converged environment**:

```text
       C---T---C              C-------C
       |   |   |              |       |
  Z1 = T---A---T         Z2 = |       |
       |   |   |              |       |
       C---T---C              C-------C

       C---T---C              C-------C
       |   |   |              |       |
  Z3 = |   |   |         Z4 = T-------T
       |   |   |              |       |
       C---T---C              C-------C
```

The partition function per site is estimated by

$$
\boxed{z=\frac{Z_1Z_2}{Z_3Z_4}}.
$$

To motivate the ratio, imagine that $C$ represents an $L\times L$ quadrant
and $T$ a strip of length $L$. The four patches have areas
$(2L+1)^2$, $(2L)^2$, $2L(2L+1)$, and $2L(2L+1)$, respectively.
Keeping only the bulk contribution $Z\sim z^{\mathrm{area}}$ gives

$$
\frac{Z_1 Z_2}{Z_3 Z_4} \simeq z^{(2L+1)^2+(2L)^2-2(2L)(2L+1)} = z.
$$

Importantly, the formula is insensitive to the normalization of $C$ and $T$.
Specifically, under $C\mapsto cC$ and $T\mapsto tT$,

$$
Z_1\mapsto c^4t^4Z_1,\quad Z_2\mapsto c^4Z_2,\quad
Z_3\mapsto c^4t^2Z_3,\quad Z_4\mapsto c^4t^2Z_4.
$$
These scale factors cancel exactly in the ratio, so no CTMRG normalization history is needed.
Rotational symmetry gives $Z_3=Z_4$, so compute this contraction once.

**Exercise 6a:** compute the patch contractions; rotational symmetry is provided.
**Exercises 6b–6c:** implement `eval_log_z` and `eval_z`.

In [ ]:
def partition_contractions(A, C, T):
    """Return Z1, Z2, Z3, Z4 from the four CTM diagrams above.

    Assumes real, reflection-symmetric tensors shared by all directions.
    Consequently the two rectangular contractions satisfy Z3 == Z4.
    """

    rho = aux_rho_1x1(C, T)

    # TODO: Contract rho with A to obtain scalar Z1.
    raise NotImplementedError("Exercise 7a: central patch.")

    # TODO: Close four corner matrices into the empty-patch scalar Z2.
    raise NotImplementedError("Exercise 7a: empty patch.")

    # TODO: Contract the rectangular patch to obtain scalar Z3.
    raise NotImplementedError("Exercise 7a: rectangular patch.")

    Z4 = Z3  # Rotational symmetry.
    return Z1, Z2, Z3, Z4


def eval_log_z(A, C, T):
    """Estimate log(z) = -beta*f from a converged isotropic CTM environment.

    The four scalar contractions are positive for the physical environment.
    Scalar rescalings of C and T cancel; a rescaling of A does not.
    """

    # TODO: Combine the four patch contractions in logarithmic form.
    raise NotImplementedError("Exercise 7b: implement eval_log_z.")


def eval_z(A, C, T):
    """Estimate z = lim Z_N**(1/N), the partition function per site."""

    # TODO: Recover z from eval_log_z.
    raise NotImplementedError("Exercise 7c: implement eval_z.")


## I.5 [Extra] Observables via source-field insertions & automatic differentiation

Magnetization, energy per site, and specific heat follow from derivatives of $\log z$.
At $H=0$,

$$
m=\frac1\beta\frac{\partial\log z}{\partial H},\qquad
u=-\frac{\partial\log z}{\partial\beta},\qquad
c_V=\frac{\partial u}{\partial T}
=\beta^2\frac{\partial^2\log z}{\partial\beta^2}.
$$

PyTorch records the operations that connect $\beta$ and $H$ to $\log z$,
including the CTMRG iterations.
The function [`torch.autograd.grad`](https://docs.pytorch.org/docs/stable/generated/torch.autograd.grad.html)
applies the chain rule through this computation graph to evaluate the derivatives automatically. In the following,
use this function to compute the observables mentioned. Two options are needed here:
- `retain_graph=True`: preserve the graph after the field derivative so it can be reused for the temperature derivatives.
- `create_graph=True`: record the first temperature derivative so it can be differentiated again.

Random boundaries can select a magnetization branch below $T_c$.
Second derivatives require careful convergence; check the CTMRG tolerance and $\chi$.

**Exercise 7:** fill the three derivative gaps in `obs_from_AD` to compute $m$ and $c_V$.
The loop over $\beta$, CTMRG call, and result collection are provided.


In [ ]:
def obs_from_AD(betas, chi, ctm_max_iter=10000, ctm_conv_tol=1e-9):
    """Return zero-field magnetization and specific heat from derivatives of log(z)."""

    m_ad = []
    cv_ad = []

    for beta_value in betas:
        H = torch.zeros((), dtype=sigma.dtype, requires_grad=True)
        beta = torch.tensor(beta_value, dtype=sigma.dtype, requires_grad=True)
        A = Ising_A(beta, H)
        C, T = ctmrg(
            A,
            *init_rand_CT(A, chi),
            chi,
            ctm_max_iter=ctm_max_iter,
            ctm_conv_tol=ctm_conv_tol,
            raise_on_nonconvergence=True,
        )
        log_z = eval_log_z(A, C, T)

        # TODO: Compute dlog_z_dH, retaining the graph for the temperature derivatives.
        raise NotImplementedError("Exercise 8: field derivative.")

        m_ad.append((dlog_z_dH / beta).item())

        # TODO: Compute dlog_z_dbeta and keep its derivative graph.
        raise NotImplementedError("Exercise 8: first temperature derivative.")

        # TODO: Differentiate dlog_z_dbeta again to obtain d2log_z_dbeta2.
        raise NotImplementedError("Exercise 8: second temperature derivative.")

        cv_ad.append((beta**2 * d2log_z_dbeta2).item())

    return m_ad, cv_ad


### I.5.1. Exact specific heat (reference)

Use the zero-field square-lattice formula below. Its logarithmic divergence
marks $\beta_c$; finite $\chi$ limits accuracy near the peak.

Reference: [Caux, Eq. (126)](https://jscaux.org/static/jsc/courses/2019_FPITFT/FROM_PI_TO_FT_2019_notes.pdf).

**Provided reference:** keep this exact formula and plotting helper unchanged.


In [ ]:
# Provided: exact specific heat and plotting helper.
def exact_specific_heat(beta):
    """Exact zero-field specific heat per site for beta > 0 and J = k_B = 1."""
    from scipy.special import ellipk, ellipe

    beta = np.asarray(beta, dtype=float)
    t = np.tanh(2 * beta)
    k = 2 * t / np.cosh(2 * beta)
    K, E = ellipk(k**2), ellipe(k**2)  # SciPy uses the parameter k**2.
    return (
        4
        / np.pi
        * (beta / t) ** 2
        * (K - E - (1 - t**2) * (np.pi / 2 + (2 * t**2 - 1) * K))
    )


def plot_specific_heat(betas, values):
    """Compare CTMRG specific heat with the exact zero-field curve."""
    betas = np.asarray(betas)
    grid = np.linspace(betas.min(), betas.max(), 400)
    fig, ax = plt.subplots(figsize=(6, 4), constrained_layout=True)
    ax.plot(grid, exact_specific_heat(grid), color="black", label="Exact")
    ax.scatter(betas, values, label="CTMRG")
    ax.axvline(
        np.log1p(np.sqrt(2)) / 2, color="gray", linestyle="--", label=r"$\beta_c$"
    )
    ax.set(xlabel=r"$\beta$", ylabel=r"$c_V$", title="Specific heat per site")
    ax.legend()
    return fig, ax


### I.5.2. Compare both observables

Sample more densely near $\beta_c$ and compare the AD results with the exact curves.

In [ ]:
# Run after completing Exercise 7.
torch.manual_seed(7)
chi=12
betas = np.concatenate(
    [np.linspace(0.3, 0.41, 5), np.linspace(0.43, 0.45, 10), np.linspace(0.48, 0.6, 5)]
)
m_ad, cv_ad = obs_from_AD(betas, chi)


In [ ]:
# Provided: compare both observables with the exact curves.
fig_m, ax_m = plot_magnetization(betas, m_ad, "Magnetization from AD")
fig_heat, ax_heat = plot_specific_heat(betas, cv_ad)

plt.show()
